<a href="https://colab.research.google.com/github/wheredawoodat949/AI-Hackathon/blob/basketball/Basketball_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basketball tracking — Phase 1 (Path A)

Mirrors `Soccer_1.ipynb`'s structure. Differences (see `CLAUDE.md` §3/§4, `sports/examples/basketball/README.md`):
- Basketball-51 has **no detection labels** — no training here, just inference.
- **One generic COCO detector** (`yolo11n.pt`, ungated, auto-downloads) instead of soccer's 3
  sport-specific checkpoints — no basketball-specific model exists yet.
- No pitch/radar mode — no basketball court keypoint model.

**Runtime > Change runtime type > T4 GPU**, then run top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone OUR repo (has sports/ vendored + the new examples/basketball/main.py) and install.
import os

REPO_URL = "https://github.com/wheredawoodat949/AI-Hackathon"
BRANCH = "basketball"

if not os.path.isdir("AI-Hackathon"):
    !git clone --branch {BRANCH} {REPO_URL}.git
%cd AI-Hackathon
!git pull
%cd sports
!pip install -q -e .
!pip install -q -r examples/basketball/requirements.txt
!pip install -q "umap-learn==0.5.6" "numba>=0.59" "numpy<2.1" "supervision==0.23.0"

In [ ]:
# Pull Basketball-51 via kagglehub (needs your Kaggle auth — interactive login or a Colab secret).
import kagglehub

path = kagglehub.dataset_download("sarbagyashakya/basketball-51-dataset")
print("Path to dataset files:", path)

In [ ]:
# Find clips (Basketball-51's exact folder layout isn't documented in-repo — glob for it).
import glob
import os

clips = []
for ext in ("mp4", "avi", "mov", "mkv"):
    clips += glob.glob(os.path.join(path, "**", f"*.{ext}"), recursive=True)
print(f"found {len(clips)} clips")
for c in clips[:10]:
    print(" ", c)

## Pick one clean clip
Basketball-51 clips are low-res (320x240) broadcast 6-second action clips. Eyeball a few and
pick one with clear, unobstructed players for the best-looking demo. Edit the index below.

In [ ]:
CLIP_INDEX = 0  # change after eyeballing a few options
source_clip = clips[CLIP_INDEX]
print("using:", source_clip)

from IPython.display import Video
Video(source_clip, embed=True, width=480)

In [ ]:
# Quick sanity check first: PLAYER_TRACKING (no UMAP/team-classifier — verified working on CPU
# in dev; this confirms the detector+tracker generalizes to real basketball footage on Colab).
%cd examples/basketball
!python main.py \
  --source_video_path "{source_clip}" \
  --target_video_path /content/basketball_tracking.mp4 \
  --device cuda --mode PLAYER_TRACKING

In [ ]:
import subprocess

subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error", "-i", "/content/basketball_tracking.mp4",
    "-vcodec", "libx264", "-pix_fmt", "yuv420p", "/content/basketball_tracking_h264.mp4",
])
from IPython.display import Video
Video("/content/basketball_tracking_h264.mp4", embed=True, width=480)

## The real Phase 1 deliverable: TEAM_CLASSIFICATION
Tracks every player + the ball AND colors players by team (SigLIP+UMAP+KMeans on crops —
`sports.common.team.TeamClassifier`, unmodified Roboflow code). This is the basketball
equivalent of Vincent's soccer demo. If this cell errors, see the troubleshooting note below.

In [ ]:
!python main.py \
  --source_video_path "{source_clip}" \
  --target_video_path /content/basketball_team.mp4 \
  --device cuda --mode TEAM_CLASSIFICATION

In [ ]:
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error", "-i", "/content/basketball_team.mp4",
    "-vcodec", "libx264", "-pix_fmt", "yuv420p", "/content/basketball_team_h264.mp4",
])
Video("/content/basketball_team_h264.mp4", embed=True, width=480)

## If TEAM_CLASSIFICATION errors
It ran cleanly for the simpler modes in local dev testing (CPU, real model). If this specific
cell crashes, it's almost certainly the UMAP/SigLIP step (`TeamClassifier.fit`), not the
detection/tracking code (already proven above). Try:
1. Pick a clip with more visible players (more crop diversity helps UMAP).
2. `pip install -U numba` and restart the runtime once, then re-run from the install cell.
3. If it still fails, `PLAYER_TRACKING` output above is still a valid demo artifact — fall back
   to that and flag this in `UPDATE.md` for follow-up.

## Next
Download `basketball_team_h264.mp4` (Colab file browser), drop it in the team Drive, and update
`UPDATE.md` on the `basketball` branch with what actually happened on this run (real frame
counts, any errors, the clip used) — per `CLAUDE.md` §6's git rules.